In [1]:
import os
import numpy as np
import pandas as pd
import torch
import hickle as hkl

from tqdm import tqdm
from numpy.linalg import norm
from sklearn.metrics import pairwise as kernel

from process_data import make_cell_embedding

BASE_DIR = "datasets/2023"

In [2]:
meta_df = pd.read_csv(os.path.join(BASE_DIR, "Model.csv"), low_memory=False)
meta_df[["OncotreeSubtype", "OncotreePrimaryDisease"]] = (
    meta_df[["OncotreeSubtype", "OncotreePrimaryDisease"]].fillna("")
)

sclc_mask = (
    meta_df["OncotreeSubtype"]
          .str.strip()
          .str.lower()
          .eq("small cell lung cancer")
)
sclc_ids = set(meta_df.loc[sclc_mask, "ModelID"])

print("sclc_ids:", len(sclc_ids))

sclc_ids: 81


In [3]:
embedding, gene_effects_df = make_cell_embedding()

print("FULL embedding shape:", embedding.shape)
print("FULL gene_effects_df shape:", gene_effects_df.shape)
print("Index match:", embedding.index.equals(gene_effects_df.index))

FULL embedding shape: (1112, 33587)
FULL gene_effects_df shape: (1112, 18435)
Index match: True


In [4]:
sclc_idx = embedding.index.intersection(list(sclc_ids))
embedding_sclc = embedding.loc[sclc_idx].copy()
gene_effects_sclc = gene_effects_df.loc[sclc_idx].copy()

print("SCLC embedding shape:", embedding_sclc.shape)
print("SCLC gene_effects_df shape:", gene_effects_sclc.shape)
assert embedding_sclc.index.equals(gene_effects_sclc.index)

SCLC embedding shape: (25, 33587)
SCLC gene_effects_df shape: (25, 18435)


In [5]:
os.makedirs("embeddings", exist_ok=True)
os.makedirs("datasets/2023", exist_ok=True)

hkl.dump(embedding_sclc, "embeddings/final_X_sclc_processed.hkl", mode="w")
hkl.dump(gene_effects_sclc, "datasets/2023/CRISPRGeneEffect_sclc_processed.hkl", mode="w")

print("Saved SCLC-only .hkl files")

pandas 2.3.3
Saved SCLC-only .hkl files


/opt/miniconda3/envs/augert/lib/python3.10/site-packages/hickle/lookup.py:1491: SerializedWarning: 'DataFrame' type not understood, data is serialized:
  warnings.warn(


In [6]:
cell_embedding = hkl.load("embeddings/final_X_sclc_processed.hkl")
cell_embedding /= norm(cell_embedding, axis=1).reshape(-1, 1)

gene_effects_df = hkl.load("datasets/2023/CRISPRGeneEffect_sclc_processed.hkl")

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_individual_rfm_cell(cell_embedding: pd.DataFrame,
                              gene_effects_df: pd.DataFrame,
                              bandwidth: float = 1.0,
                              reg: float = 1e-5,
                              device: torch.device = device) -> torch.Tensor:
    X = torch.tensor(cell_embedding.values, device=device).float()

    dists_np = kernel.euclidean_distances(cell_embedding.values, cell_embedding.values)
    cell_distances = torch.tensor(dists_np, device=device).float()
    cell_distances.fill_diagonal_(0)

    y = torch.tensor(gene_effects_df.values, device=device).float()

    K = torch.exp(-bandwidth * (cell_distances)**0.5)
    sol = torch.linalg.solve(K + reg * torch.eye(K.shape[0], device=device), y)
    return sol  # (n_cells, n_knockouts)

sol = train_individual_rfm_cell(cell_embedding, gene_effects_df, bandwidth=1.0, reg=1e-5, device=device)
print("sol shape:", tuple(sol.shape))

sol shape: (25, 18435)


In [8]:
def euclidean_distances(samples, centers, M=None, squared=True, diag_only=False):
    if M is None:
        samples_norm = torch.sum(samples**2, dim=1, keepdim=True)
    else:
        if diag_only:
            samples_norm = (samples * M) * samples
        else:
            samples_norm = (samples @ M) * samples
        samples_norm = torch.sum(samples_norm, dim=1, keepdims=True)

    if samples is centers:
        centers_norm = samples_norm
    else:
        if M is None:
            centers_norm = torch.sum(centers**2, dim=1, keepdims=True)
        else:
            if diag_only:
                centers_norm = (centers * M) * centers
            else:
                centers_norm = (centers @ M) * centers
            centers_norm = torch.sum(centers_norm, dim=1, keepdims=True)
    centers_norm = torch.reshape(centers_norm, (1, -1))

    distances = samples.mm(torch.t(centers))
    distances.mul_(-2)
    distances.add_(samples_norm)
    distances.add_(centers_norm)
    if not squared:
        distances.clamp_(min=0)
        distances.sqrt_()
    return distances

In [9]:
def laplace_kernel(samples, centers, bandwidth, M=None, diag_only=False):
    kernel_mat = euclidean_distances(samples, centers, M=M, squared=False, diag_only=diag_only)
    kernel_mat.clamp_(min=0)
    gamma = 1. / bandwidth
    kernel_mat.mul_(-gamma)
    kernel_mat.exp_()
    return kernel_mat

In [10]:
def get_grads(X, sol_T, P, L=1, centering=False, diag_only=True):
    K = laplace_kernel(X, X, bandwidth=1, M=P, diag_only=diag_only)

    dist = euclidean_distances(X, X, M=P, squared=False, diag_only=diag_only)
    dist.clamp_(min=0)
    dist[dist < 1e-10] = 0

    K = K / torch.where(dist == 0, torch.ones_like(dist), dist)
    K[torch.isinf(K)] = 0.0

    n, d = X.shape
    num_kos, n2 = sol_T.shape
    assert n == n2

    grads = torch.zeros((d, num_kos), device=X.device)
    for i in tqdm(range(num_kos)):
        weight = sol_T[i, :].reshape((-1, 1))
        step2 = K @ (weight * X)
        step3 = (weight.T @ K).T * X
        G = (step2 - step3) * (-1 / L)
        G = torch.sum(G**2, axis=0)
        grads[:, i] = G / n
    return grads

In [11]:
X_t = torch.tensor(cell_embedding.values, device=device).float()
n, d = X_t.shape
P = torch.ones(d, device=device).double()

grads_t = get_grads(X_t, sol.T, L=1, P=P, centering=False, diag_only=True)
print("grads tensor shape:", tuple(grads_t.shape))  # (d, num_kos)

100%|██████████| 18435/18435 [00:54<00:00, 338.11it/s]

grads tensor shape: (33587, 18435)


In [12]:
def get_pcc(cell_embedding: pd.DataFrame, gene_effects_df: pd.DataFrame) -> pd.DataFrame:
    exp_cols = [c for c in cell_embedding.columns if c.split("_")[-1] == "exp"]

    std_val = cell_embedding[exp_cols].std(axis=0).replace(0, 1)
    z = (cell_embedding[exp_cols] - cell_embedding[exp_cols].mean(axis=0)) / std_val

    cell_embedding = cell_embedding.copy()
    cell_embedding[exp_cols] *= (np.abs(z) < 3).fillna(0).astype(int)

    normalized_cell_embedding = cell_embedding - cell_embedding.mean(axis=0)
    normalized_gene_effects_df = gene_effects_df - gene_effects_df.mean(axis=0)

    cell_norms = (normalized_cell_embedding**2).sum(axis=0).values
    gene_norms = (normalized_gene_effects_df**2).sum(axis=0).values

    pcc = (normalized_cell_embedding.T @ normalized_gene_effects_df) / (
        cell_norms.reshape((-1, 1)) @ gene_norms.reshape((1, -1))
    )**0.5

    return pd.DataFrame(pcc, columns=normalized_gene_effects_df.columns, index=normalized_cell_embedding.columns)

In [13]:
features = cell_embedding.columns
knockouts = gene_effects_df.columns

grads_df = pd.DataFrame(
    grads_t.detach().cpu().numpy(),
    index=features,
    columns=knockouts
)

In [14]:
pcc = get_pcc(cell_embedding, gene_effects_df).fillna(0)
pcc = pcc.loc[grads_df.index]

mut = [x for x in pcc.index if x.split("_")[-1] != "exp"]
pcc.loc[mut] = -(pcc.loc[mut].clip(upper=0))

exp = [x for x in pcc.index if x.split("_")[-1] == "exp"]
pcc.loc[exp] = abs(pcc.loc[exp])

feature_importance_df = grads_df * pcc
print("feature_importance_df shape:", feature_importance_df.shape)

feature_importance_df shape: (33587, 18435)


In [15]:
os.makedirs("datasets", exist_ok=True)

np.save("datasets/feature_importances_sclc_data.npy", feature_importance_df.values)
np.save("datasets/feature_importances_sclc_index.npy", feature_importance_df.index.to_numpy())
np.save("datasets/feature_importances_sclc_columns.npy", feature_importance_df.columns.to_numpy())

print("Saved SCLC-only feature importance outputs")

Saved SCLC-only feature importance outputs
